In [1]:
# install / upgrade required packages
# run this cell first if packages are missing
# !pip install transformers datasets scikit-learn torch accelerate
import warnings
warnings.filterwarnings("ignore")
print("imports ready!")


imports ready!


In [2]:
import pandas as pd

# load the dataset
df = pd.read_csv("../dataset/sarcasm_dataset_multilingual_10000.csv")

# quick look
print("Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nLabel distribution:")
print(df["label"].value_counts())


Shape: (10000, 2)

First 5 rows:
                                 text  label
0                    Wow you are busy      1
1                 Kay mast you forgot      1
2       Rehne do tu call ka nahi kela      1
3      Haan obviously you didn't call      1
4  Ho na, tum toh tum call nahi karte      1

Label distribution:
label
1    5000
0    5000
Name: count, dtype: int64


In [3]:
# cleaning the text - nothing fancy, just lowercase + strip
def clean_text(text):
    text = str(text)
    text = text.lower().strip()
    return text

df["text"] = df["text"].apply(clean_text)

# check
print("After cleaning:")
print(df.head())
print("\nAny nulls?", df.isnull().sum())


After cleaning:
                                 text  label
0                    wow you are busy      1
1                 kay mast you forgot      1
2       rehne do tu call ka nahi kela      1
3      haan obviously you didn't call      1
4  ho na, tum toh tum call nahi karte      1

Any nulls? text     0
label    0
dtype: int64


In [4]:
from sklearn.model_selection import train_test_split

# split: 80% train, 10% val, 10% test
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"])

print("Train size:", len(train_df))
print("Val size:", len(val_df))
print("Test size:", len(test_df))


Train size: 8000
Val size: 1000
Test size: 1000


In [5]:
from transformers import AutoTokenizer

# using distilbert-multilingual because it supports Hindi + Marathi + Hinglish
model_name = "distilbert-base-multilingual-cased"

print("Loading tokenizer...")
tok = AutoTokenizer.from_pretrained(model_name)
print("Done!")

# test the tokenizer real quick
sample = "Wow tum call nahi karte"
tokens = tok(sample, return_tensors="pt")
print("\nSample tokenized shape:", tokens["input_ids"].shape)


Loading tokenizer...
Done!

Sample tokenized shape: torch.Size([1, 10])


In [6]:
import torch
from torch.utils.data import Dataset

class SarcasmDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        # tokenize each text
        encoded = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        
        # return as flat dict
        item = {key: val.squeeze(0) for key, val in encoded.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# create the three datasets
train_dataset = SarcasmDataset(train_df["text"], train_df["label"], tok)
val_dataset   = SarcasmDataset(val_df["text"],   val_df["label"],   tok)
test_dataset  = SarcasmDataset(test_df["text"],  test_df["label"],  tok)

print("Train dataset size:", len(train_dataset))
print("Val dataset size:", len(val_dataset))
print("Test dataset size:", len(test_dataset))


Train dataset size: 8000
Val dataset size: 1000
Test dataset size: 1000


In [7]:
from transformers import AutoModelForSequenceClassification

# 2 labels: 0 = normal, 1 = sarcasm
print("Loading model...")
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
print("Model loaded!")
print("Total params:", sum(p.numel() for p in model.parameters()) // 1_000_000, "M")


Loading model...


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded!
Total params: 135 M


In [8]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# this function is called by the Trainer to compute metrics after each eval
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)     # pick the highest probability class
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "f1": f1}

print("compute_metrics function ready!")


compute_metrics function ready!


In [11]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="../sarcasm_model",

    learning_rate=3e-5,                 # slightly higher for faster learning
    per_device_train_batch_size=16,     # smaller batch (CPU friendly)
    per_device_eval_batch_size=16,

    num_train_epochs=2,                 # reduce epochs
    warmup_steps=50,
    weight_decay=0.01,

    logging_steps=50,

    # CPU optimized
    no_cuda=True,                      # force CPU
    dataloader_num_workers=0,          # avoid multiprocessing overhead

    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("🚀 Starting training (CPU optimized)...")
trainer.train()
print("✅ Training done!")

🚀 Starting training (CPU optimized)...


  5%|▌         | 50/1000 [01:41<32:14,  2.04s/it]

{'loss': 0.6133, 'grad_norm': 3.538292169570923, 'learning_rate': 3e-05, 'epoch': 0.1}


 10%|█         | 100/1000 [03:33<31:42,  2.11s/it]

{'loss': 0.0243, 'grad_norm': 0.03870643302798271, 'learning_rate': 2.8421052631578946e-05, 'epoch': 0.2}


...

{'loss': 0.0001, 'grad_norm': 0.0018010104540735483, 'learning_rate': 0.0, 'epoch': 2.0}


                                                   
100%|██████████| 1000/1000 [34:58<00:00,  2.01s/it]

{'eval_loss': 6.7652523284778e-05, 'eval_accuracy': 1.0, 'eval_f1': 1.0, 'eval_runtime': 27.9233, 'eval_samples_per_second': 35.812, 'eval_steps_per_second': 2.256, 'epoch': 2.0}


100%|██████████| 1000/1000 [35:00<00:00,  2.10s/it]

{'train_runtime': 2100.3969, 'train_samples_per_second': 7.618, 'train_steps_per_second': 0.476, 'train_loss': 0.03217340849898755, 'epoch': 2.0}
✅ Training done!


In [12]:
# evaluate on the test set (unseen data)
print("Evaluating on test set...")
test_results = trainer.evaluate(test_dataset)

print("\n=== TEST RESULTS ===")
for k, v in test_results.items():
    print(f"{k}: {round(v, 4)}")


Evaluating on test set...


100%|██████████| 63/63 [00:27<00:00,  2.33it/s]


=== TEST RESULTS ===
eval_loss: 0.0001
eval_accuracy: 1.0
eval_f1: 1.0
eval_runtime: 27.6269
eval_samples_per_second: 36.197
eval_steps_per_second: 2.28
epoch: 2.0


In [13]:
# save the trained model and tokenizer  
# then we can use it directly in models.py
save_path = "../sarcasm_model"

model.save_pretrained(save_path)
tok.save_pretrained(save_path)

print(f"Model saved to: {save_path}")
print("Now you can load it with:")
print(f'  pipeline("text-classification", model="{save_path}")')


Model saved to: ../sarcasm_model
Now you can load it with:
  pipeline("text-classification", model="../sarcasm_model")


In [ ]:
from transformers import pipeline

# load back the saved model
sarcasm_pipe = pipeline("text-classification", model="../sarcasm_model", tokenizer="../sarcasm_model")

# test with some real Hinglish messages
test_msgs = [
    "Haan obviously you didn't call",         
    "Wow tumne reply nahi kiya",               
    "okay thanks for letting me know",        
    "I am fine, how are you?",                
    "Rehne do tu mala ignore karto",          
]

print("=== PREDICTIONS ===")
for msg in test_msgs:
    result = sarcasm_pipe(msg)[0]
    label_name = "SARCASM" if result["label"] == "LABEL_1" else "NORMAL"
    print(f"\nMsg: {msg}")
    print(f"  → {label_name} ({result['score']:.2%} confident)")


=== PREDICTIONS ===

Msg: Haan obviously you didn't call
  → SARCASM (99.99% confident)

Msg: Wow tumne reply nahi kiya
  → SARCASM (99.99% confident)

Msg: okay thanks for letting me know
  → SARCASM (99.99% confident)

Msg: I am fine, how are you?
  → NORMAL (99.97% confident)

Msg: Rehne do tu mala ignore karto
  → SARCASM (99.99% confident)


In [15]:
# ensemble: combine model + rule-based signals for better accuracy
# this is the secret boost mentioned in Phase 6

# sarcastic keywords in Hinglish / English
sarc_keywords = [
    "obviously", "of course", "wow", "great", "perfect", "sure",
    "haan haan", "haan obviously", "bilkul", "acha thik hai",
    "kay mast", "rehne do", "ho na", "vahh", "wah"
]

def smart_sarcasm_detection(text, sarcasm_pipeline):
    text_lower = text.lower()
    
    # rule 1: check keywords
    rule_detected = any(kw in text_lower for kw in sarc_keywords)
    
    # rule 2: model prediction
    result = sarcasm_pipeline(text)[0]
    model_conf = result["score"]
    model_label = result["label"]

    # ensemble logic
    if rule_detected:
        return "sarcasm"
    elif model_label == "LABEL_1" and model_conf > 0.7:
        return "sarcasm"
    else:
        return "no sarcasm"

# test it
for msg in test_msgs:
    ans = smart_sarcasm_detection(msg, sarcasm_pipe)
    print(f"{msg}\n  → {ans}\n")


Haan obviously you didn't call
  → sarcasm

Wow tumne reply nahi kiya
  → sarcasm

okay thanks for letting me know
  → sarcasm

I am fine, how are you?
  → no sarcasm

Rehne do tu mala ignore karto
  → sarcasm

